# RAG Corpus Data Quality Audit

## tl;dr

**Decision: do not promote versioned hybrid RAG to the production corpus yet.** The legacy database has 1,117 document rows, but only 357 ready/enabled rows with chunks and just 4 distinct chunk-content groups. All 357 chunked rows belong to duplicate groups, Chroma contains no legacy vectors, and the embedding preflight is unavailable. The evidence supports a controlled cleanup/rebuild rehearsal, not production rollout.


## Context & Methods

This notebook consumes two privacy-bounded aggregate JSON files: an application-side profile and an independent database-side reconciliation. Neither source emits document names, paths, content, or opaque hashes.

### Key Assumptions

- A logical content group is defined by the SHA-256 hash of each document's chunks ordered by ordinal and row id.
- `ready` and enabled documents are the retrieval-eligible legacy population.
- Source-file availability means the current workspace can resolve the original file without exposing its path here.
- A production rollout requires schema revision 0019, usable embeddings, and a benchmark broader than four logical cases.


In [1]:
from pathlib import Path
import json

root = Path.cwd()
profile_path = root / 'data/analysis/rag-data-quality-report-source-v2-20260802.json'
validation_path = root / 'data/analysis/rag-data-quality-validation-source-v2-20260802.json'
profile = json.loads(profile_path.read_text(encoding="utf-8"))
validation = json.loads(validation_path.read_text(encoding="utf-8"))

assert validation["all_checks_match"] is True
assert profile["grain"]["logical_content_groups"] == validation["sql_results"]["chunk_manifest_groups"]
print("Loaded aggregate-only sources; independent checks match.")


Loaded aggregate-only sources; independent checks match.


## Data

The analysis grain is a document row, a legacy chunk row, or a logical chunk-content group. Counts below are exact snapshots of the local `personal_assistant` schema; percentages use the ready/enabled population unless labeled otherwise.


In [2]:
grain = profile["grain"]
completeness = profile["completeness"]
uniqueness = profile["uniqueness"]
integrity = profile["integrity"]
vectors = profile["vector_integrity"]

funnel = [
    ("All document rows", grain["document_rows"]),
    ("Ready and enabled", grain["ready_enabled_documents"]),
    ("Ready/enabled with chunks", grain["ready_enabled_with_chunks"]),
    ("Source-resolvable with chunks", round(grain["ready_enabled_with_chunks"] * completeness["source_file_available_with_chunks_rate"])),
]
print("stage | rows | share of all rows")
print("--- | ---: | ---:")
for stage, rows in funnel:
    print(f"{stage} | {rows:,} | {rows / grain['document_rows']:.1%}")


stage | rows | share of all rows
--- | ---: | ---:
All document rows | 1,117 | 100.0%
Ready and enabled | 383 | 34.3%
Ready/enabled with chunks | 357 | 32.0%
Source-resolvable with chunks | 32 | 2.9%


## Results

The retrieval-eligible corpus is numerically large but logically tiny. The independent SQL path reproduces all five decision-critical counts and shows that every usable document contains exactly one chunk.


In [3]:
group_sizes = validation["sql_results"]["manifest_group_sizes"]
checks = validation["checks"]
print("metric | application profile | independent SQL | match")
print("--- | ---: | ---: | :---:")
for metric, values in checks.items():
    print(f"{metric.replace('_', ' ')} | {values['profile']:,} | {values['sql']:,} | {'yes' if values['matches'] else 'no'}")
print()
print("logical group rank | document rows")
print("---: | ---:")
for rank, size in enumerate(group_sizes, start=1):
    print(f"{rank} | {size:,}")


metric | application profile | independent SQL | match
--- | ---: | ---: | :---:
document rows | 1,117 | 1,117 | yes
legacy chunk rows | 358 | 358 | yes
ready enabled documents | 383 | 383 | yes
ready enabled with chunks | 357 | 357 | yes
chunk manifest groups | 4 | 4 | yes

logical group rank | document rows
---: | ---:
1 | 180
2 | 59
3 | 59
4 | 59


In [4]:
quality_rows = [
    ("Duplicate rate among chunked documents", f"{uniqueness['duplicate_document_rate_among_chunked']:.1%}"),
    ("Excess duplicate document rows", f"{uniqueness['excess_duplicate_documents']:,}"),
    ("BM25 rows missing", f"{integrity['bm25_missing_chunks']:,}"),
    ("Documents with chunk-count mismatch", f"{integrity['chunk_count_mismatch_documents']:,}"),
    ("Legacy vector coverage", f"{vectors['coverage_rate']:.1%}"),
    ("Valid declared hash coverage on chunked documents", f"{completeness['valid_content_hash_with_chunks_rate']:.1%}"),
]
print("quality signal | result")
print("--- | ---:")
for label, value in quality_rows:
    print(f"{label} | {value}")


quality signal | result
--- | ---:
Duplicate rate among chunked documents | 100.0%
Excess duplicate document rows | 353
BM25 rows missing | 54
Documents with chunk-count mismatch | 76
Legacy vector coverage | 0.0%
Valid declared hash coverage on chunked documents | 0.0%


## Takeaways

1. **Hold production rollout.** Schema revision 0012, zero vector coverage, unavailable embedding preflight, and four logical cases cannot support a defensible hybrid-RAG promotion gate.
2. **Preserve before cleaning.** Keep the verified pre-upgrade clone and perform any deduplication only as a dry-run or on an isolated clone.
3. **Rebuild by logical content.** Select one canonical document per chunk manifest, repair chunk metadata/BM25 text, and regenerate vectors after the embedding service passes preflight.
4. **Require human-reviewed breadth.** Expand the benchmark beyond the four generated logical cases before comparing legacy and versioned retrieval.

### Open Questions

- Are the four repeated content groups expected fixtures, accidental repeated imports, or the wrong database/data directory for production use?
- Should documents without resolvable source files be retained as legacy-only evidence, exported, or quarantined?

### Caveats

- The 357 chunked documents have no valid declared content hash, so declared hashes cannot independently validate the chunk-manifest partition.
- The report measures the local snapshot only; it does not infer user intent from private document contents.
- No production database rows were changed during this audit.
